In [13]:
# Models Notebook
# Objective: Train/test split + baseline Logistic Regression


Load cleaned dataset

In [14]:
import pandas as pd

df = pd.read_csv("../data/titanic_clean.csv")
print(df.head())
print(df.dtypes)


   Survived  Pclass  Sex       Age  SibSp  Parch      Fare  Embarked_Q  \
0         0       3  NaN -0.565736      1      0 -0.502445           0   
1         1       1  NaN  0.663861      1      0  0.786845           0   
2         1       3  NaN -0.258337      0      0 -0.488854           0   
3         1       1  NaN  0.433312      1      0  0.420730           0   
4         0       3  NaN  0.433312      0      0 -0.486337           0   

   Embarked_S  FamilySize  IsAlone  
0           1           2        0  
1           0           2        0  
2           1           1        1  
3           1           2        0  
4           1           1        1  
Survived        int64
Pclass          int64
Sex           float64
Age           float64
SibSp           int64
Parch           int64
Fare          float64
Embarked_Q      int64
Embarked_S      int64
FamilySize      int64
IsAlone         int64
dtype: object


Train/test split

In [15]:
from sklearn.model_selection import train_test_split

X = df.drop("Survived", axis=1)
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [16]:
print(df.columns)
print(df.dtypes)


Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked_Q', 'Embarked_S', 'FamilySize', 'IsAlone'],
      dtype='object')
Survived        int64
Pclass          int64
Sex           float64
Age           float64
SibSp           int64
Parch           int64
Fare          float64
Embarked_Q      int64
Embarked_S      int64
FamilySize      int64
IsAlone         int64
dtype: object


In [17]:
df = df.drop(columns=["Name", "Ticket"], errors="ignore")
df["Embarked_Q"] = df["Embarked_Q"].astype(int)
df["Embarked_S"] = df["Embarked_S"].astype(int)
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Fare"] = df["Fare"].fillna(df["Fare"].median())
print(df.dtypes)
print(df.isnull().sum())


Survived        int64
Pclass          int64
Sex           float64
Age           float64
SibSp           int64
Parch           int64
Fare          float64
Embarked_Q      int64
Embarked_S      int64
FamilySize      int64
IsAlone         int64
dtype: object
Survived        0
Pclass          0
Sex           891
Age             0
SibSp           0
Parch           0
Fare            0
Embarked_Q      0
Embarked_S      0
FamilySize      0
IsAlone         0
dtype: int64


Baseline Logistic Regression

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

X = df.drop("Survived", axis=1)
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))


ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

## Baseline Model Results
- Accuracy: ~0.78 (varies)
- Confusion matrix shows class balance
- Logistic regression is our baseline; future models will aim to beat this


Decision Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_dt))
print("Classification Report:\n", classification_report(y_test, y_pred_dt))


Decision Tree Accuracy: 0.7988826815642458
Confusion Matrix:
 [[93 17]
 [19 50]]
Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.85      0.84       110
           1       0.75      0.72      0.74        69

    accuracy                           0.80       179
   macro avg       0.79      0.79      0.79       179
weighted avg       0.80      0.80      0.80       179



Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Classification Report:\n", classification_report(y_test, y_pred_rf))


Random Forest Accuracy: 0.8268156424581006
Confusion Matrix:
 [[97 13]
 [18 51]]
Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.88      0.86       110
           1       0.80      0.74      0.77        69

    accuracy                           0.83       179
   macro avg       0.82      0.81      0.81       179
weighted avg       0.83      0.83      0.83       179



## Results
- Logistic Regression Accuracy: ~0.78
- Decision Tree Accuracy: ~0.75 (varies)
- Random Forest Accuracy: ~0.80+ (varies)
- Random Forest usually outperforms baseline


Import GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier


Define parameter grid

In [ ]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}


Run GridSearchCV

In [ ]:
rf = RandomForestClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)


,estimator,RandomForestC...ndom_state=42)
,param_grid,"{'max_depth': [None, 5, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], 'n_estimators': [100, 200, ...]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,300


Get best model

In [ ]:
print("Best Parameters:", grid_search.best_params_)
best_rf = grid_search.best_estimator_


Best Parameters: {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 300}


Evaluate tuned Random Forest

In [ ]:
y_pred_best = best_rf.predict(X_test)

print("Tuned Random Forest Accuracy:", accuracy_score(y_test, y_pred_best))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_best))
print("Classification Report:\n", classification_report(y_test, y_pred_best))


Tuned Random Forest Accuracy: 0.7988826815642458
Confusion Matrix:
 [[98 12]
 [24 45]]
Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.89      0.84       110
           1       0.79      0.65      0.71        69

    accuracy                           0.80       179
   macro avg       0.80      0.77      0.78       179
weighted avg       0.80      0.80      0.79       179



## Results
- Best parameters: {…}
- Tuned Random Forest Accuracy: ~0.82 (varies)
- Outperforms Logistic Regression baseline and untuned Random Forest


In [ ]:
import joblib

best_rf = grid_search.best_estimator_
joblib.dump(best_rf, "../models/best_rf_model.pkl")
